In [10]:
import polars as pl
import numpy as np
import datetime as dt
import os
import sys
import json 
import importlib

sys.path.append("../utils/")

import helpers as hp

In [11]:
rng = np.random.default_rng(seed=274)

In [12]:

with open("../configs/zone_distances.json", "r") as json_file:
    zone_distances = json.load(json_file)

In [13]:
zone_distances

{'Yerbabuena': {'centroids': [20.964404421308334, -101.2847459818324],
  'points': [{'point': [20.9743, -101.300626],
    'distance': 0.01871089133756122},
   {'point': [20.946278, -101.298483], 'distance': 0.022743632462390598},
   {'point': [20.951706, -101.264939], 'distance': 0.023527992541503697},
   {'point': [20.979447, -101.274624], 'distance': 0.018131014585796808},
   {'point': [20.985938, -101.285636], 'distance': 0.02155196379935847},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122}]},
 'Marfil': {'centroids': [20.998877268148007, -101.29015919031652],
  'points': [{'point': [20.979212, -101.292779],
    'distance': 0.019839006379117525},
   {'point': [20.991242, -101.304647], 'distance': 0.016376628136369357},
   {'point': [21.006043, -101.294156], 'distance': 0.008205010702043177},
   {'point': [21.019272, -101.283363], 'distance': 0.021497285645706604},
   {'point': [21.008196, -101.275124], 'distance': 0.017688858391175503},
   {'point': [20.999863,

In [14]:
gym_hours = {"open" : "6.0",
             "close" : "22.0"}

profiles = {
"1": {
"name" : "frequent_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .89},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .76},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .85},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .7},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.9},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .9},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "6.43",
"hour_mean" : ".75",
"plan_probability" : {"1": 0.05, "2": 0.2, "3" : 0.45, "4": 0.3}
},
"2" : {
"name" : "frequent_noon",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .7},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .82},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .67},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .87},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.78},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .84},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "13.16",
"hour_mean" : "1.40",
"plan_probability" : {"1": .2, "2": 0.5, "3" : 0.25, "4": 0.05}
            },
"3" :{
"name" : "frequent_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .74},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .87},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .76},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .84},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "19.5",
"hour_mean" : "1.15",
"plan_probability" : {"1": 0.15, "2": 0.15, "3" : 0.45, "4": 0.25}
},
"4" :{
"name" : "random_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .4},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .4},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .3},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 0.5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" : .6},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : 1},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "8.5",
"hour_mean" : "2.05",
"plan_probability" : {"1": 0.65, "2": 0.25, "3" : 0.1, "4": 0}
},
"5" :{
"name" : "random_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .3},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .3},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .6},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.9},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "20.1",
"hour_mean" : "2.08",
"plan_probability" : {"1": 0.4, "2": 0.3, "3" : 0.2, "4": 0.1}
},
"6" :{
"name" : "full_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .5},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .5},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.5},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .5},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "14.00",
"hour_mean" : "3.0",
"plan_probability" : {"1": 0.59, "2": 0.3, "3" : 0.1, "4": 0.01}
},
"7" :{
"name" : "rare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .2},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .4},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .2},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.3},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .4},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "16.0",
"hour_mean" : "3.5",
"plan_probability" : {"1": 0.85, "2": 0.1, "3" : 0.05, "4": 0}
},
"8" :{
"name" : "ultrarare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .1},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .1},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.2},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "14.5",
"hour_mean" : "5.5",
"plan_probability" : {"1": 0.9, "2": 0.08, "3" : 0.02, "4": 0}
}
}

profile_weights = {"1" : 0.23076923, "2": 0.12820513, "3" : 0.25641026, "4" : 0.07692308, "5" : 0.07692308, "6" : 0.05128205, "7" : 0.07692308, "8" : 0.1025641}

for profile in profiles.keys():
    print(profile)
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
        
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    
    profiles[profile]["total_weight"] = sum_weight

    assert 1 == sum([i for i in profiles[profile]["plan_probability"].values() ])
profiles

1
2
3
4
5
6
7
8


{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.89,
    'day_probability': 0.178},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.76,
    'day_probability': 0.152},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.85,
    'day_probability': 0.16999999999999998},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.7,
    'day_probability': 0.13999999999999999},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': '.75',
  'plan_probability': {'1': 0.05, '2': 0.2, '3': 0.45, '4': 0.3},
  'total_weight': 5.0},
 '2': {'name': 'frequent_noon',
  'days_pro

In [15]:
# Functions
def create_customer_profiles(profile_weights_ : dict, n_):
    return {str(i): str(int(rng.choice( list(profile_weights.keys()), p = list(profile_weights.values()) ) ) ) for i in range(n_) }

In [16]:
create_customer_profiles(profiles, 9)

{'0': '8',
 '1': '1',
 '2': '3',
 '3': '7',
 '4': '3',
 '5': '4',
 '6': '1',
 '7': '5',
 '8': '1'}

In [17]:
# prob = np.array((.9,0.5,1,.3,.3,.2,.3,.4))

# prob/sum(prob)

In [18]:
importlib.reload(hp)

oli =hp.create_customer(zone_distances)

Creating customer
36254


In [19]:
oli

{'id': None,
 'name': 'Hilda Elvira',
 'last_name': 'Lira Santacruz',
 'gender': np.False_,
 'birth_date': datetime.datetime(1993, 9, 7, 0, 0),
 'lat': 20.98821795202615,
 'lon': -101.29917077337164,
 'zipcode': 36254,
 'email': 'g**************r@gmail.com',
 'phone_number': '(473)8821424',
 'created_at': datetime.datetime(2026, 5, 22, 18, 51, 28, 618619),
 'status': 'Active',
 'profile_type': None,
 'updated_at': datetime.datetime(2026, 5, 22, 18, 51, 28, 618625)}

In [ ]:
date = '2026-01-02'

def create_customer_access_data(date_, profile_metadata_, gym_hours_, data_path_ = "../data/", customers_file_ = "customers.csv", payment_file_ = "payments.csv", access_file_ = "access.csv", plans_file_ = "suscription_plans.csv"):
    
    opening_date = "2026-01-02"

    assert date_ >= opening_date, f"Gym doesn't exists before {opening_date}"

    # Define Schemas
    customers_schema = {"Customer_Id" :pl.Int64,
                        "Name": pl.String,
                        "Last_Name": pl.String,
                        "Gender" : pl.Boolean,
                        "Birth_Date" : pl.Datetime,
                        "Latitude": pl.Float32,
                        "Longitude" : pl.Float32,
                        "Zipcode" : pl.Int64 ,
                        "Email" : pl.String,
                        "Phone_Number" : pl.String,
                        "Created_At" : pl.Datetime,
                        "Status" : pl.String,
                        "Profile_Type" : pl.String,
                        "Updated_At" : pl.Datetime}
    

    payments_schema = {"Payment_Id" : pl.String,
                       "Plan_Id" : pl.Int64,
                       "Customer_Id" : pl.Int64,
                       "Payment_Amount" : pl.Float16,
                       "Payment_Time" : pl.Datetime,
                       "Plan_Expiration_Time" : pl.Datetime,
                       "Payment_Status": pl.String,
                       "Updated_At" : pl.Datetime
    }

    access_schema = {"Visit_Id" : pl.Int64,
                     "Customer_Id" : pl.Int64,
                     "Plan_Id" : pl.Int64,
                     "Branch_Id" : pl.Int8,
                     "Payment_Id" : pl.String,
                     "Date" : pl.Datetime,
                     "Updated_At" : pl.Datetime
    }

    suscription_plan_schema = {"Plan_Id" : pl.Int64,
                                "Plan_Name" : pl.String,
                                "Plan_Cost" : pl.Float16, 
                                "Plan_Duration" : pl.Int64,
                                "Plan_Start_Time" : pl.Datetime, 
                                "Plan_End_Time" : pl.Datetime, 
                                "Plan_Description" :pl.String,
                                "Updated_At": pl.Datetime
                                }
    
    eval_date = dt.datetime.strptime(date_, "%Y-%m-%d")
    day_names_dic = {0 : 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4:'Friday', 5:'Saturday', 6: 'Sunday'}
    day_of_week = eval_date.weekday()
    day_name = day_names_dic[day_of_week]

    print(eval_date, day_of_week, day_name)
    
    if not os.path.isdir(data_path_):
        os.makedirs(data_path_, exist_ok = True)

    if not os.path.exists(data_path_ + customers_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = customers_schema,
                                                    orient="row").write_csv(data_path_+ customers_file_)
    
    if not os.path.exists(data_path_ + payment_file_):
        print("Creating payments file")
        df_payments = pl.DataFrame([], schema = payments_schema,
                                                    orient="row").write_csv(data_path_+ payment_file_)
        
    if not os.path.exists(data_path_ + access_file_):
        print("Creating access registry file")
        df_access = pl.DataFrame([], schema = access_schema,
                                                    orient="row").write_csv(data_path_+ access_file_)

    df_customers = pl.read_csv(data_path_+ customers_file_, schema=customers_schema)
    df_access = pl.read_csv(data_path_+ access_file_, schema=access_schema)
    df_plans = pl.read_csv(data_path_ + plans_file_, schema = suscription_plan_schema)
    df_payments = pl.read_csv(data_path_+ payment_file_, schema=payments_schema)
    
    #########################
    # 1. Create new customers
    #########################

    # 1.1. Customers configs
    tot_new = int(abs(rng.normal(0, 1)))
    print("Total new customers", tot_new)

    max_id = df_customers.select(pl.max("Customer_Id")).item()

    counter = max_id
    if max_id == None:
        counter = 0
        tot_new = 5

    counter += 1 

    dict_profiles = create_customer_profiles(profile_metadata_, tot_new)

    # 1.2 Generate customers
    new_customers = []
    new_customers_ids = []
    for new_cus in range(tot_new):

        new_customer = hp.create_customer(zone_distances)
        new_customer["id"] = counter
        new_customer["profile_type"] = dict_profiles[str(new_cus)]
        new_customer["created_at"] = dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())

        new_customers.append(tuple(new_customer.values()))
        new_customers_ids.append(counter)
        counter +=1

    # 1.3 Save file
    df_new_customers = pl.DataFrame(new_customers, customers_schema, orient = "row")
    df_customers = pl.concat([df_customers, df_new_customers])
    #df_customers.write_csv(data_path_ + customers_file_)

    ###########################
    # 2. Create Access Registry
    ###########################

    # 2.1 Given at least one customer, generate random visits
    if df_customers.shape[0] >0:
        temp_cus = df_customers.filter(pl.col("Status") =="Active").to_dicts()

        payments = []
        access = []
        access_count = 1
        for row in temp_cus:
            cus_id = row["Customer_Id"]
            profile = str(row["Profile_Type"])
            profile_data = profile_metadata_[profile]
            day_proba  = profile_data["days_probability"][day_name[0:3].lower()]["day_probability"]

            has_visit = rng.choice([1,0], p = [day_proba, 1-day_proba])

            if max_id == None:
                has_visit = 1

            if cus_id in new_customers_ids: # If it's a new customer, has to enter gym
                has_visit = 1

            print(f"Customer {cus_id} with profile {profile} appeared {bool(has_visit)}")

            # Check is customer assisted
            if has_visit == 0:
                continue
            
            ######################
            # 2.2. Create Payments
            ######################

            df_payments = pl.read_csv(data_path_+ payment_file_, schema=payments_schema)
            temp_pay = df_payments.filter(pl.col("Customer_Id") == cus_id).sort(by = ["Plan_Expiration_Time"], descending=[True])

            payment_count = temp_pay.shape[0]
            expiration_date = temp_pay.select(pl.max("Plan_Expiration_Time")).item()

            # Generate payment if conditions are meet
            if (payment_count == 0) or (expiration_date < dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())):
                
                plan_choice = int(rng.choice( list(profile_metadata_[profile]["plan_probability"].keys()), p= list(profile_metadata_[profile]["plan_probability"].values())))

                if cus_id in new_customers_ids: # If it's a new customer, has to enter gym
                    plan_choice = 1

                print(f"Customer {cus_id} is paying plan {plan_choice}")

                temp_plan  = df_plans.filter(pl.col("Plan_Id")== plan_choice)

                amount = temp_plan.select(pl.col("Plan_Cost")).item()
                duration = temp_plan.select(pl.col("Plan_Duration")).item()
                payment_count +=1
                payments_dict = {"Payment_Id" : str(cus_id) + "-" + str(payment_count),
                                "Plan_Id" : plan_choice,
                                "Customer_Id" : cus_id,
                                "Payment_Amount" : amount,
                                "Payment_Time" : dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time()),
                                "Plan_Expiration_Time" : dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time()) + dt.timedelta(days = duration),
                                "Payment_Status": "Completed",
                                "Updated_At" : dt.datetime.now()
                }

                payments.append(tuple(payments_dict.values()))

            
                df_new_payments = pl.DataFrame(payments, schema = payments_schema, orient = "row")
                df_payments = pl.concat([df_payments, df_new_payments])
            #df_payments.write_csv(data_path_+ payment_file_)
            
            # Generate access
            df_access = pl.read_csv(data_path_+ access_file_, schema=access_schema)
            access_dict = {"Visit_Id" : access_count,
                    "Customer_Id" : cus_id,
                    "Plan_Id" : plan_choice,
                    "Branch_Id" : 1,
                    "Payment_Id" : str(cus_id) + "-" + str(payment_count),
                    "Date" : dt.datetime.strptime(date, "%Y-%m-%d"),
                    "Updated_At" : dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())
                    }

            access_count += 1

            access.append(tuple(access_dict.values()))

            df_new_accesss = pl.DataFrame(access, schema = access_schema, orient = "row")
            df_access = pl.concat([df_access, df_new_accesss])
            #df_access.write_csv(data_path_+ access_file_)

    return df_customers, df_payments, df_access
    

datasets = create_customer_access_data(date, profiles, gym_hours)

2026-01-02 00:00:00 4 Friday
Total new customers 2
Creating customer
36253


In [59]:
display(datasets[0])
display(datasets[1])
display(datasets[2])

Customer_Id,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,datetime[μs],f32,f32,i64,str,str,datetime[μs],str,str,datetime[μs]
1,"""Mario Emilio""","""Pedraza Brito""",true,1985-06-08 00:00:00,21.008369,-101.257607,36089,"""p****************z@gmail.com""","""(473)6648176""",2026-01-02 20:30:24.040978,"""Active""","""3""",2026-05-22 20:30:24.040936
2,"""Gloria Elisa""","""Almaraz Echeverría""",false,2005-12-10 00:00:00,21.003372,-101.26194,36255,"""g**********q@hotmail.com""","""(473)7579182""",2026-01-02 20:30:27.623511,"""Active""","""2""",2026-05-22 20:30:27.623468


Payment_Id,Plan_Id,Customer_Id,Payment_Amount,Payment_Time,Plan_Expiration_Time,Payment_Status,Updated_At
str,i64,i64,f16,datetime[μs],datetime[μs],str,datetime[μs]
"""1-1""",3,1,1200.0,2026-01-02 20:30:27.625245,2026-02-01 20:30:27.625256,"""Completed""",2026-05-22 20:30:27.625260
"""2-1""",1,2,150.0,2026-01-02 20:30:27.626134,2026-01-03 20:30:27.626142,"""Completed""",2026-05-22 20:30:27.626146


Visit_Id,Customer_Id,Plan_Id,Branch_Id,Payment_Id,Date,Updated_At
i64,i64,str,i8,str,datetime[μs],datetime[μs]
1,1,"""3""",1,"""1-1""",2026-01-02 00:00:00,2026-01-02 20:30:27.625502
2,2,"""1""",1,"""2-1""",2026-01-02 00:00:00,2026-01-02 20:30:27.626353


In [ ]:
for profile in profiles.keys():
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
    
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    profiles[profile]["total_weight"] = sum_weight
profiles
        

{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.89,
    'day_probability': 0.178},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.76,
    'day_probability': 0.152},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.85,
    'day_probability': 0.16999999999999998},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.7,
    'day_probability': 0.13999999999999999},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': '.75',
  'total_weight': 5.0},
 '2': {'name': 'frequent_noon',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
   

In [ ]:
dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())

datetime.datetime(2026, 1, 1, 11, 0, 34, 763406)

In [ ]:
my_dic = {}


for i in range(100):
    value = int(abs(rng.normal(0, 1.5)))
    my_dic[value] = my_dic.get(value, 0) + 1

my_dic

{2: 8, 0: 52, 1: 36, 3: 3, 4: 1}

In [61]:
8*650/5

1040.0